In [3]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np

In [4]:
torch.cuda.is_available()
#df = pd.read_csv("C38_ready-forSure.csv")
df = pd.read_csv("RTN_test.csv")

# features = ['x', 'y', 'z', 'vx', 'vy', 'vz', 'days_offset']
# targets = ['x_error', 'y_error', 'z_error']

features = ['x', 'y', 'z', 'vx', 'vy', 'vz', 'days_offset']
targets = ['r_error', 't_error', 'n_error']

X = df[features].values
y = df[targets].values

#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
split_idx = int(len(df) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

df_test = df.iloc[split_idx:].reset_index(drop=True)

scaler_X = StandardScaler()
X_train = scaler_X.fit_transform(X_train)
X_test = scaler_X.transform(X_test)

scaler_y = StandardScaler()
y_train = scaler_y.fit_transform(y_train)
y_test = scaler_y.transform(y_test)

# X_tensor = torch.tensor(X, dtype=torch.float32)
# y_tensor = torch.tensor(y, dtype=torch.float32)
# dataset = TensorDataset(X_tensor, y_tensor)

train_dataset = TensorDataset(  
    torch.tensor(X_train, dtype=torch.float32), 
    torch.tensor(y_train, dtype=torch.float32)
)
test_dataset = TensorDataset(
    torch.tensor(X_test, dtype=torch.float32), 
    torch.tensor(y_test, dtype=torch.float32)
)

train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=64, shuffle=False)


In [5]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(7, 128),
            nn.GELU(),
            nn.Linear(128, 128),
            nn.GELU(),
            nn.Linear(128,128),
            nn.ReLU(),
            nn.Linear(128, 3)
        )

    def forward(self, x):
        return self.network(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Model().to(device=device)

loss_fn = nn.L1Loss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-2)

epochs = 1000
for epoch in range(epochs):
    model.train()
    ov_loss = 0.0

    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        y_pred = model(batch_X)
        loss = loss_fn(y_pred, batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        ov_loss += loss.item() * batch_X.size(0)

    epoch_loss = ov_loss / len(train_dataset)

    if (epoch + 1) % 100 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss:.4f}")

Epoch [100/1000], Loss: 0.1505
Epoch [200/1000], Loss: 0.1072
Epoch [300/1000], Loss: 0.0441
Epoch [400/1000], Loss: 0.0468
Epoch [500/1000], Loss: 0.0289
Epoch [600/1000], Loss: 0.0221
Epoch [700/1000], Loss: 0.0213
Epoch [800/1000], Loss: 0.0206
Epoch [900/1000], Loss: 0.0201
Epoch [1000/1000], Loss: 0.0185


In [6]:
model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        y_pred = model(batch_X)
        all_preds.append(y_pred.cpu().numpy())
        all_targets.append(batch_y.cpu().numpy())

preds_array = np.vstack(all_preds)
targets_array = np.vstack(all_targets)

preds_km = scaler_y.inverse_transform(preds_array)
targets_km = scaler_y.inverse_transform(targets_array)

err3d_model = np.linalg.norm(preds_km - targets_km, axis=1)
err3d_raw_sgp4 = np.linalg.norm(targets_km, axis=1)

df_test['Err3D_Model'] = err3d_model
df_test['Err3D_SGP4'] = err3d_raw_sgp4

print("\n=== PODSUMOWANIE OGÓLNE 3D [km] ===")
print(f"SGP4  -> Średnia: {err3d_raw_sgp4.mean():.3f} | Mediana: {np.median(err3d_raw_sgp4):.3f} | 90th perc (ogon): {np.percentile(err3d_raw_sgp4, 90):.3f}")
print(f"Model -> Średnia: {err3d_model.mean():.3f} | Mediana: {np.median(err3d_model):.3f} | 90th perc (ogon): {np.percentile(err3d_model, 90):.3f}")

print("\n=== BŁĄD 3D W ROZBICIU NA DAYS_OFFSET [km] ===")
summary = df_test.groupby('days_offset').apply(lambda g: pd.Series({
    'Liczba_Próbek': len(g),
    'Mediana_SGP4': g['Err3D_SGP4'].median(),
    'Mediana_Model': g['Err3D_Model'].median(),
    'Średnia_SGP4': g['Err3D_SGP4'].mean(),
    'Średnia_Model': g['Err3D_Model'].mean(),
    '90perc_SGP4': np.percentile(g['Err3D_SGP4'], 90),
    '90perc_Model': np.percentile(g['Err3D_Model'], 90)
}))

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
print(summary.round(3).to_string())


=== PODSUMOWANIE OGÓLNE 3D [km] ===
SGP4  -> Średnia: 6.159 | Mediana: 1.491 | 90th perc (ogon): 3.639
Model -> Średnia: 6.544 | Mediana: 1.835 | 90th perc (ogon): 4.040

=== BŁĄD 3D W ROZBICIU NA DAYS_OFFSET [km] ===
             Liczba_Próbek  Mediana_SGP4  Mediana_Model  Średnia_SGP4  Średnia_Model  90perc_SGP4  90perc_Model
days_offset                                                                                                    
0                    106.0         1.253          1.622         1.398          1.848        2.439         2.938
1                    107.0         1.117          1.663         3.103          3.580        2.518         2.844
3                    107.0         1.331          1.869         5.669          6.236        2.154         3.360
5                    108.0         2.937          2.773        14.346         14.394        4.605         5.811


C:\Users\kozak\AppData\Local\Temp\ipykernel_6148\2958093815.py:29: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  summary = df_test.groupby('days_offset').apply(lambda g: pd.Series({


In [7]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

# Szybki test na obecnych danych
X_train, X_test, y_train, y_test = train_test_split(X, y[:, 0], test_size=0.2, shuffle=False) # Chronologicznie!

gbr = GradientBoostingRegressor(n_estimators=100, random_state=42)
gbr.fit(X_train, y_train)
preds = gbr.predict(X_test)

print("MAE Gradient Boosting dla osi X:", mean_absolute_error(y_test, preds))

MAE Gradient Boosting dla osi X: 1.0202757114188867


In [8]:
N = 10  # ile ostatnich wierszy pokazać

view = pd.DataFrame({
    'target_epoch':  df_test['target_epoch'].values,
    'days_offset':   df_test['days_offset'].values,

    # prawdziwy błąd SGP4 (prawda_SP3 - SGP4), bez korekty
    'SGP4_dx': targets_km[:, 0],
    'SGP4_dy': targets_km[:, 1],
    'SGP4_dz': targets_km[:, 2],

    # korekta przewidziana przez model
    'Model_dx': preds_km[:, 0],
    'Model_dy': preds_km[:, 1],
    'Model_dz': preds_km[:, 2],

    # odległość od prawdy w 3D: bez korekty vs z korektą
    'Dist_SGP4 [km]':  np.linalg.norm(targets_km, axis=1),
    'Dist_Model [km]': np.linalg.norm(targets_km - preds_km, axis=1),
})
view['Poprawa [km]'] = view['Dist_SGP4 [km]'] - view['Dist_Model [km]']  # >0 = model pomógł

pd.set_option('display.width', 1000)
pd.set_option('display.max_columns', None)
print(view.tail(N).round(3).to_string(index=False))

       target_epoch  days_offset    SGP4_dx    SGP4_dy     SGP4_dz  Model_dx  Model_dy  Model_dz  Dist_SGP4 [km]  Dist_Model [km]  Poprawa [km]
2025-11-15 12:00:00            0   0.128000  -0.585000    0.517000     1.046    -0.870     0.874        0.791000         1.026000        -0.235
2025-11-16 12:00:00            1   0.134000  -0.351000    0.308000     0.528    -0.855     0.668        0.486000         0.734000        -0.248
2025-11-18 12:00:00            3   0.491000   0.269000   -0.121000    -0.871     1.397    -0.460        0.572000         1.801000        -1.228
2025-11-20 12:00:00            5 -26.643000 161.126999 -109.703003    -1.616     4.622    -2.607      196.740005       191.285004         5.455
2025-11-16 12:00:00            0   0.190000  -0.638000    0.544000     0.863    -0.755     0.742        0.860000         0.710000         0.149
2025-11-17 12:00:00            1   0.308000  -0.508000    0.446000     0.250    -0.686     0.479        0.743000         0.190000       

In [9]:
view = pd.DataFrame({
    'tle_epoch':   df_test['tle_epoch'].values,
    'target_epoch': df_test['target_epoch'].values,
    'days_offset': df_test['days_offset'].values,

    # surowy wynik SGP4 (pozycja, km)
    'SGP4_x': df_test['x'].values,
    'SGP4_y': df_test['y'].values,
    'SGP4_z': df_test['z'].values,

    # surowy wynik modelu (przewidziany błąd, km)
    'Model_x': preds_km[:, 0],
    'Model_y': preds_km[:, 1],
    'Model_z': preds_km[:, 2],
})
print(view.tail(10).round(3).to_string(index=False))

                 tle_epoch        target_epoch  days_offset     SGP4_x    SGP4_y     SGP4_z  Model_x  Model_y  Model_z
2025-11-15 15:12:38.515968 2025-11-15 12:00:00            0 -22415.789 23219.704 -27149.081    1.046   -0.870    0.874
2025-11-15 15:12:38.515968 2025-11-16 12:00:00            1 -22557.298 23519.061 -26774.411    0.528   -0.855    0.668
2025-11-15 15:12:38.515968 2025-11-18 12:00:00            3 -22807.374 24135.913 -26008.223   -0.871    1.397   -0.460
2025-11-15 15:12:38.515968 2025-11-20 12:00:00            5 -23013.322 24775.808 -25218.758   -1.616    4.622   -2.607
2025-11-16 17:57:22.499712 2025-11-16 12:00:00            0 -22557.565 23518.997 -26774.154    0.863   -0.755    0.742
2025-11-16 17:57:22.499712 2025-11-17 12:00:00            1 -22688.276 23824.419 -26393.790    0.250   -0.686    0.479
2025-11-16 17:57:22.499712 2025-11-21 12:00:00            5 -23100.964 25103.588 -24814.036   -1.498    4.687   -2.636
2025-11-19 14:15:02.708640 2025-11-20 12:00:00  

In [10]:
for a in [0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0]:
    e = np.linalg.norm(targets_km - a * preds_km, axis=1)
    print(a, e.mean().round(3), np.median(e).round(3), np.percentile(e, 90).round(3))

0 6.159 1.491 3.639
0.25 6.091 1.443 3.426
0.5 6.143 1.466 3.491
0.75 6.302 1.645 3.704
1.0 6.544 1.835 4.04
1.5 7.165 2.577 4.891
2.0 7.876 3.343 6.229
